# Sheikh Reference-Match and Visual Description POC

This notebook uses the shared `FaceIdentifier` gallery to find known people in a test image. It returns a simple `people_result` such as `{"people": ["abdul_rahman_al_sudais", "mohammed_bin_salman"]}`, then passes the matched names and their face-match provenance to the configured visual-description model. The model is not asked to identify faces; it receives names already matched against the local reference gallery.

In [ ]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'arabic_poc' / 'sheikh_poc.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the RAG-Anything project directory.')
sys.path.insert(0, str(PROJECT_ROOT.resolve()))
load_dotenv(PROJECT_ROOT / '.env', override=True)

gallery_setting = Path(os.getenv('FACE_GALLERY_PATH', 'reference_faces'))
REFERENCE_DIR = gallery_setting if gallery_setting.is_absolute() else PROJECT_ROOT / gallery_setting
TEST_IMAGE = PROJECT_ROOT / 'data' / 'visual_samples' / 'mbs.jpeg'
FACE_MODEL = os.getenv('FACE_MODEL', 'buffalo_l')
FACE_MATCH_THRESHOLD = float(os.getenv('FACE_MATCH_THRESHOLD', '0.5'))
QWEN_VL_MODEL = os.getenv('QWEN_VL_MODEL', 'models/qwen2.5-vl-3b')

if not TEST_IMAGE.is_file():
    raise FileNotFoundError(f'Choose an existing test image: {TEST_IMAGE}')
if not REFERENCE_DIR.is_dir():
    raise FileNotFoundError(f'Create the reference gallery first: {REFERENCE_DIR}')

from arabic_poc.face_identifier import FaceIdentifier
from arabic_poc.models import image_parser
from arabic_poc.remote import enabled

face_identifier = FaceIdentifier(
    REFERENCE_DIR,
    model_name=FACE_MODEL,
    threshold=FACE_MATCH_THRESHOLD,
    min_face_size=int(os.getenv('MIN_FACE_SIZE', '40')),
    min_detection_score=float(os.getenv('MIN_DETECTION_SCORE', '0.6')),
)
vision = image_parser(QWEN_VL_MODEL, describe=True)
provider = 'remote ' + os.getenv('REMOTE_VISION_MODEL', '') if enabled('vision') else 'local Qwen2.5-VL'
print(f'Gallery: {REFERENCE_DIR}')
print(f'Face model: {FACE_MODEL}; threshold: {FACE_MATCH_THRESHOLD}')
print(f'Visual model: {provider}')

Face model loaded. Kernel: /Users/ushnah.abbasi/RAG-Anything/.venv/bin/python
Reference gallery: /Users/ushnah.abbasi/RAG-Anything/reference_faces


In [ ]:
import json
from arabic_poc.sheikh_poc import describe_with_matches

result = describe_with_matches(TEST_IMAGE, vision, face_identifier)
people_result = {'people': result['people']}

print('People result:')
print(json.dumps(people_result, ensure_ascii=False, indent=2))
print('\nVisual description result:')
print(json.dumps({
    'description': result['description'],
    'entities': result['entities'],
    'person_matches': result['person_matches'],
}, ensure_ascii=False, indent=2))

Found 1 usable face(s) in /Users/ushnah.abbasi/RAG-Anything/data/visual_samples/mbs.jpeg

Matching 1 face(s)...

Face 1: best candidate = mohammed_bin_salman, similarity = 0.7331
MATCH FOUND: mohammed_bin_salman (similarity=0.7331)

FINAL RESULT
Face 1: mohammed_bin_salman (similarity=0.7331)


## Notes

- Add multiple reference photos under `reference_faces/<canonical_person_name>/`.
- Configure `FACE_MATCH_THRESHOLD` and calibrate it against held-out known and unknown images.
- `people_result` is the clean name-only JSON for retrieval workflows; the full result retains scores and `face_recognition` provenance.
- Matched names are supplied to the visual model as reference-gallery context and merged into `entities.people`; the VLM is not asked to identify or confirm faces.
- The POC only recognizes people represented in the local gallery. Review matches before treating them as reliable.